### Import the dataset

In [20]:
import pandas as pd
df = pd.read_csv("notebook/spam.csv", encoding="ISO-8859-1")

In [21]:
df.head()

,v1,v2,Unnamed: 2,Unnamed: 3,Unnamed: 4
0,ham,"Go until jurong point, crazy.. Available only ...",NaN,NaN,NaN
1,ham,Ok lar... Joking wif u oni...,NaN,NaN,NaN
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...,NaN,NaN,NaN
3,ham,U dun say so early hor... U c already then say...,NaN,NaN,NaN
4,ham,"Nah I don't think he goes to usf, he lives aro...",NaN,NaN,NaN


In [22]:
df.dtypes

v1            str
v2            str
Unnamed: 2    str
Unnamed: 3    str
Unnamed: 4    str
dtype: object

In [23]:
df.isnull().sum()

v1               0
v2               0
Unnamed: 2    5522
Unnamed: 3    5560
Unnamed: 4    5566
dtype: int64

### Drop Columns with null values 

The dataset contains some extra columns such as `Unnamed: 2`, `Unnamed: 3`, and `Unnamed: 4` which are not useful for our spam detection model.

These columns may contain missing or irrelevant data, so we remove them to keep only the important features.

After dropping these columns, we inspect the dataset structure using `df.info()` to ensure the dataset is clean.

In [24]:
df = df.drop(['Unnamed: 2', 'Unnamed: 3', 'Unnamed: 4'], axis=1)
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 5572 entries, 0 to 5571
Data columns (total 2 columns):
 #   Column  Non-Null Count  Dtype
---  ------  --------------  -----
 0   v1      5572 non-null   str  
 1   v2      5572 non-null   str  
dtypes: str(2)
memory usage: 541.4 KB


### 3. Rename Columns

For better readability, let's rename the columns `v1` and `v2` to `target` and `text` respectively.

In [25]:
df = df.rename(columns={'v1': 'target', 'v2': 'text'})
print("Columns renamed successfully!")
display(df.head())
df.info()

Columns renamed successfully!


,target,text
0,ham,"Go until jurong point, crazy.. Available only ..."
1,ham,Ok lar... Joking wif u oni...
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...
3,ham,U dun say so early hor... U c already then say...
4,ham,"Nah I don't think he goes to usf, he lives aro..."


<class 'pandas.DataFrame'>
RangeIndex: 5572 entries, 0 to 5571
Data columns (total 2 columns):
 #   Column  Non-Null Count  Dtype
---  ------  --------------  -----
 0   target  5572 non-null   str  
 1   text    5572 non-null   str  
dtypes: str(2)
memory usage: 541.4 KB


### 4. Convert Text to Lowercase

It's a good practice in text processing to convert all text to lowercase to ensure consistency and avoid treating 'Word' and 'word' as different tokens.

In [26]:
df['text'] = df['text'].str.lower()
print("Text column converted to lowercase successfully!")
display(df.head())

Text column converted to lowercase successfully!


,target,text
0,ham,"go until jurong point, crazy.. available only ..."
1,ham,ok lar... joking wif u oni...
2,spam,free entry in 2 a wkly comp to win fa cup fina...
3,ham,u dun say so early hor... u c already then say...
4,ham,"nah i don't think he goes to usf, he lives aro..."


### 5. Remove Punctuation

Removing punctuation helps in normalizing the text data, making it easier for algorithms to process. We'll use regular expressions to achieve this.

In [27]:
import re

def remove_punctuation(text):
    return re.sub(r'[^a-zA-Z0-9]', ' ', text)

df['text'] = df['text'].apply(remove_punctuation)
print("Punctuation removed from text column successfully!")
display(df.head())

Punctuation removed from text column successfully!


,target,text
0,ham,go until jurong point crazy available only ...
1,ham,ok lar joking wif u oni
2,spam,free entry in 2 a wkly comp to win fa cup fina...
3,ham,u dun say so early hor u c already then say
4,ham,nah i don t think he goes to usf he lives aro...


### Stopwords Removal

In this step, we remove common English stopwords (such as "is", "the", "and", "to") that do not add meaningful information for spam detection.

We use Scikit-learn’s built-in `ENGLISH_STOP_WORDS` list to filter out these words from each message.

After removing stopwords, the cleaned text is stored in a new column called **`cleaned`**, which will be used for feature extraction (TF-IDF) in the next step.

In [28]:
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS

def remove_stopwords(text):
    words = text.split()
    filtered = [word for word in words if word not in ENGLISH_STOP_WORDS]
    return " ".join(filtered)

df["cleaned"] = df["text"].apply(remove_stopwords)
print("Stopwords removed and stored in 'cleaned' column successfully!")
display(df.head())

Stopwords removed and stored in 'cleaned' column successfully!


,target,text,cleaned
0,ham,go until jurong point crazy available only ...,jurong point crazy available bugis n great wor...
1,ham,ok lar joking wif u oni,ok lar joking wif u oni
2,spam,free entry in 2 a wkly comp to win fa cup fina...,free entry 2 wkly comp win fa cup final tkts 2...
3,ham,u dun say so early hor u c already then say,u dun say early hor u c say
4,ham,nah i don t think he goes to usf he lives aro...,nah don t think goes usf lives


## Feature Extraction using TF-IDF

In this step, we convert the cleaned text data into numerical features using **TF-IDF (Term Frequency - Inverse Document Frequency)**.

TF-IDF helps the model understand the importance of each word in a message by giving higher weight to important words and reducing the weight of commonly used words.

We configure the vectorizer with:
- **max_features=5000** → limits vocabulary size to the top 5000 most important words
- **stop_words='english'** → removes common English stopwords automatically

After transformation:
- `X` contains the numerical feature matrix
- `y` contains the target labels (spam or ham)

In [29]:
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer(
    max_features=5000,
    stop_words='english'
)

X = vectorizer.fit_transform(df["cleaned"])
y = df["target"]

In [30]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42
)

### Model Training

In this step, we train machine learning models on the processed dataset to classify messages as **Spam** or **Ham**.

We use the TF-IDF feature matrix (`X`) as input and the target labels (`y`) as output.

Different classification algorithms are tested (such as Naive Bayes and Support Vector Machine) to compare their performance and select the best-performing model.

The model will learn patterns from the training data and will be evaluated on unseen test data to measure its accuracy and generalization ability.

## Naive Bayes

In [31]:
from sklearn.naive_bayes import MultinomialNB

model = MultinomialNB()
model.fit(X_train, y_train)

,"alpha alpha: float or array-like of shape (n_features,), default=1.0Additive (Laplace/Lidstone) smoothing parameter(set alpha=0 and force_alpha=True, for no smoothing).",1.0
,"force_alpha force_alpha: bool, default=TrueIf False and alpha is less than 1e-10, it will set alpha to1e-10. If True, alpha will remain unchanged. This may causenumerical errors if alpha is too close to 0... versionadded:: 1.2.. versionchanged:: 1.4 The default value of `force_alpha` changed to `True`.",True
,"fit_prior fit_prior: bool, default=TrueWhether to learn class prior probabilities or not.If false, a uniform prior will be used.",True
,"class_prior class_prior: array-like of shape (n_classes,), default=NonePrior probabilities of the classes. If specified, the priors are notadjusted according to the data.",None


In [32]:
y_pred = model.predict(X_test)

In [33]:
from sklearn.metrics import accuracy_score, classification_report

print("Accuracy:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred))

Accuracy: 0.9757847533632287
              precision    recall  f1-score   support

         ham       0.97      1.00      0.99       965
        spam       1.00      0.82      0.90       150

    accuracy                           0.98      1115
   macro avg       0.99      0.91      0.94      1115
weighted avg       0.98      0.98      0.97      1115



## Support Vector Machine (SVM)

In [34]:
from sklearn.svm import SVC

# Initialize and train the SVM model
svm_model = SVC(random_state=42)
svm_model.fit(X_train, y_train)
print("SVM model trained successfully!")

SVM model trained successfully!


In [35]:
# Make predictions on the test set
y_pred_svm = svm_model.predict(X_test)
print("Predictions made using SVM model!")

Predictions made using SVM model!


In [36]:
# Evaluate the SVM model
from sklearn.metrics import accuracy_score, classification_report

print("SVM Accuracy:", accuracy_score(y_test, y_pred_svm))
print("\nSVM Classification Report:")
print(classification_report(y_test, y_pred_svm))

SVM Accuracy: 0.9757847533632287

SVM Classification Report:
              precision    recall  f1-score   support

         ham       0.97      1.00      0.99       965
        spam       0.99      0.83      0.90       150

    accuracy                           0.98      1115
   macro avg       0.98      0.91      0.94      1115
weighted avg       0.98      0.98      0.97      1115



## Logistic Regression

In [37]:
from sklearn.linear_model import LogisticRegression

# Initialize and train the Logistic Regression model
log_reg_model = LogisticRegression(max_iter=1000, random_state=42)
log_reg_model.fit(X_train, y_train)
print("Logistic Regression model trained successfully!")

Logistic Regression model trained successfully!


In [38]:
# Make predictions on the test set
y_pred_lr = log_reg_model.predict(X_test)
print("Predictions made using Logistic Regression model!")

Predictions made using Logistic Regression model!


In [39]:
# Evaluate the Logistic Regression model
from sklearn.metrics import accuracy_score, classification_report

print("Logistic Regression Accuracy:", accuracy_score(y_test, y_pred_lr))
print("\nLogistic Regression Classification Report:")
print(classification_report(y_test, y_pred_lr))

Logistic Regression Accuracy: 0.9488789237668162

Logistic Regression Classification Report:
              precision    recall  f1-score   support

         ham       0.95      1.00      0.97       965
        spam       0.97      0.64      0.77       150

    accuracy                           0.95      1115
   macro avg       0.96      0.82      0.87      1115
weighted avg       0.95      0.95      0.94      1115



### Saving the Trained Model and Vectorizer

After training and evaluating the model, we save it for future use so we do not need to retrain it again.

We use Python’s `joblib` library to store:
- The trained **SVM model**
- The **TF-IDF vectorizer**

Both files are saved in the `models/` directory.

This allows us to load the model later and make predictions on unseen data without repeating the training process.

In [40]:
import joblib
import os

# Save models locally in project folder
model_save_path = 'models/'

os.makedirs(model_save_path, exist_ok=True)

print(f"Saving models to: {model_save_path}")

joblib.dump(svm_model, model_save_path + "spam_model.pkl")
joblib.dump(vectorizer, model_save_path + "vectorizer.pkl")

print("Models saved.")
print(os.listdir(model_save_path))

Saving models to: models/
Models saved.
['spam_model.pkl', 'vectorizer.pkl']
